In [ ]:
import geopandas as gpd
import pandas as pd
from tqdm import tqdm
import numpy as np
import shapely

# Read full gpkg
file_path = 'EZG_Gewaesser.gpkg'
catchments = gpd.read_file(file_path, layer='Teileinzugsgebiet').set_index('EZGNR')

In [ ]:
bad_catchments = [
    115980, 165528,
    100660, 174677,
    121316, 188528,
    100399, 101433,
    117505, 136966,
    161473, 174062,
    181851, 183156
]

replace_pairs = [
    (1, 0),
    (3, 2),
    (4, 5),
    (8, 6),
    (11, 7),
    (12, 9),
    (13, 10),
]


In [ ]:
catchlist = {}

for s in tqdm(catchments.index, desc='Processing polygons'):
    # Find all upstream catchments.
    ind_upstr_s = np.argwhere(
        (catchments['H1'] >= catchments.loc[s]['H1']) &
        (catchments['H1'] < catchments.loc[s]['H2'])
    )
    # Convert to actual indices.
    ind_upstr_s = list(catchments.index[list(ind_upstr_s.flatten())])

    # If no upstream, add itself.
    if len(ind_upstr_s) == 0:
        catchlist[s] = []

    # If upstream, add them and optionally replace loops.
    else:
        # Get replace replace_pairs index if current catchment must be replaced.
        replace_i = [i for (i, pair) in enumerate(replace_pairs) if s == bad_catchments[pair[0]]]

        # Only if catchment needs to be replaced.
        if len(replace_i) > 0:
            # Replace `replace_source` with `replace_target`.
            replace_source, replace_target = [bad_catchments[i] for i in replace_pairs[replace_i[0]]]

            # Overwrite `s` with new catchment index `replace_target`.
            s = replace_target

        # Add key to catchlist if it is not there yet. this is necessary because we add to the same key multiple
        # times if we replace a catchment.
        if s not in catchlist:
            catchlist[s] = []

        # This is the candidate list of upstream catchments.
        update_list_temp = catchlist[s] + list(ind_upstr_s)

        # This is the actual list of upstream catchments, which we create in the following loop.
        update_list = []
        for el in update_list_temp:
            # Get replace replace_pairs index if current catchment must be replaced.
            replace_i = [i for (i, pair) in enumerate(replace_pairs) if el == bad_catchments[pair[0]]]

            # Only if catchment needs to be replaced.
            if len(replace_i) > 0:
                # Replace `replace_source` with `replace_target`.
                replace_source, replace_target = [bad_catchments[i] for i in replace_pairs[replace_i[0]]]

                # If the element is `replace_source` , we replace it with `replace_target`.
                el = replace_target

            # We make sure that there are no duplicates and we exclude self.
            if (el not in update_list) and (el != s):
                update_list.append(el)

        # Replace the key with the actual upstream catchment list.
        catchlist[s] = update_list


In [ ]:
downstream_links = {}

# Iterate catchments and upstream neighbors.
for catch_id, upstream_ids in tqdm(catchlist.items(), desc='Processing connectivity'):

    all_next_upstream = []

    for upstream_id in upstream_ids:
        all_next_upstream.extend(catchlist[upstream_id])

    all_next_upstream = np.unique(all_next_upstream)

    for upstream_id in upstream_ids:
        if upstream_id not in all_next_upstream:
            downstream_links[upstream_id] = catch_id


In [ ]:
out_of_ch_ids = [
    107276, 174166, 155239, 164653, 173507, 196736, 155220, 104701,
    168587, 101474, 107176, 100597, 100195, 101908, 103895, 126977]

catchments_ch = catchments.loc[catchments.index.difference(out_of_ch_ids)]
catchments_ch_merged = catchments_ch.dissolve()

In [ ]:
camels = gpd.read_file('./camels_ch/catchment_delineations/CAMELS_CH_catchments.shp')
camels['gauge_id'] = camels['gauge_id'].astype(int)
camels = camels.set_index('gauge_id')

catchments_merged = catchments.dissolve()

In [ ]:
# Adapted from the answer by Fmess (https://stackoverflow.com/a/71966323) to
# https://stackoverflow.com/questions/65073549/combine-and-sum-values-of-overlapping-polygons-in-geopandas,
# licensed under CC BY-SA 4.0 (https://creativecommons.org/licenses/by-sa/4.0/).
camels_lines = shapely.ops.unary_union(shapely.ops.linemerge([geom for geom in camels.exterior if geom is not None]))
polygons = list(shapely.ops.polygonize(camels_lines))
intersects = gpd.GeoDataFrame({'geometry': polygons})
intersects['Shape_Area'] = intersects.geometry.area
intersects = intersects.sort_values(by='Shape_Area', ascending=False).iloc[:len(camels)]
intersects = intersects.set_crs('EPSG:2056')

In [ ]:
cdiff = intersects.overlay(catchments_ch_merged, how='difference').explode().reset_index()
cdiff['Shape_Area'] = cdiff.geometry.area

In [ ]:
camels_diff = cdiff.loc[cdiff['Shape_Area'] > 10000]

In [ ]:
catchments_ch['downstream_ezgnr'] = None

for ezgnr, downsteam_ezgnr in downstream_links.items():
    if ezgnr not in catchments_ch.index:
        continue

    if catchments_ch.loc[ezgnr, 'geometry'] is not None:
        catchments_ch.loc[ezgnr, 'downstream_ezgnr'] = downsteam_ezgnr

catchments_ch = catchments_ch[catchments_ch.geometry.notnull()]

In [ ]:
camels_diff.index = range(200000, 200000 + len(camels_diff))
camels_diff.index.name = 'EZGNR'
camels_diff = camels_diff.drop(columns=['level_0', 'level_1'])

catchments_ch['origin'] = 'ezg'
camels_diff['origin'] = 'camels'

catchments_combined = pd.concat((catchments_ch, camels_diff))
catchments_combined['Shape_Area'] =  catchments_combined.geometry.area

In [ ]:
catchments_combined.to_file('./comb_ctch/catchments.shp')